In [ ]:
import os
import re
import pdfplumber
import nltk
import json
import csv
from pathlib import Path
from nltk.tokenize import sent_tokenize
from google.colab import files
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

nltk.download('punkt_tab')

# 📂 Set up directories
input_dir = "your_input_directory"
output_jsonl = "osha_rag_dataset.jsonl"
output_csv = "osha_rag_dataset.csv"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/5

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
from google import genai

client = genai.Client(api_key="your_GEMINI_API_KEY")
chat = client.chats.create(model="gemini-2.0-flash")

In [ ]:
# 🧠 RAG Chunker
def chunk_text(text, chunk_size=500, overlap=50):
    sents = sent_tokenize(text)
    chunks = []
    current_chunk = []

    for sent in sents:
        current_chunk.append(sent)
        if sum(len(s) for s in current_chunk) >= chunk_size:
            chunks.append(" ".join(current_chunk))
            current_chunk = current_chunk[-1 * overlap // 20:]  # retain overlap

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

# 🏗️ Main extractor
rag_chunks = []

for file_path in Path(input_dir).rglob("*.pdf"):
    try:
        with pdfplumber.open(file_path) as pdf:
            file_text = ""
            for i, page in enumerate(pdf.pages):
                text = page.extract_text()
                if text:
                    text = re.sub(r'[\s]+', ' ', text)  # normalize whitespace
                    file_text += f"\n=== Page {i+1} ===\n{text}"

            chunks = chunk_text(file_text)

            for idx, chunk in enumerate(chunks):
                rag_chunks.append({
                    "chunk_id": f"{file_path.stem}_{idx}",
                    "source_file": file_path.name,
                    "chunk_text": chunk.strip(),
                })
    except Exception as e:
        print(f"Error processing {file_path.name}: {e}")

# 💾 Save JSONL
with open(output_jsonl, "w", encoding="utf-8") as f:
    for row in rag_chunks:
        json.dump(row, f)
        f.write("\n")

# 💾 Save CSV (optional)
with open(output_csv, "w", newline='', encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["chunk_id", "source_file", "chunk_text"])
    writer.writeheader()
    writer.writerows(rag_chunks)

# 📥 Download
files.download(output_jsonl)
files.download(output_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 📥 Load JSONL chunked OSHA data
jsonl_path = "Osha_GuidelineLLM/osha_rag_dataset.jsonl"  # make sure this file exists in Google Drive, can change to osha_rag_index.faiss if retrain locally
chunks = []
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

# 🧠 Load sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 🔢 Generate embeddings for each chunk
texts = [chunk["chunk_text"] for chunk in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

# 🧠 Create FAISS vector index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# 📄 Create metadata mapping (e.g. for retrieval explanations)
chunk_id_to_metadata = {i: chunks[i] for i in range(len(chunks))}

# 💾 Save FAISS index and metadata
faiss.write_index(index, "osha_rag_index.faiss")
with open("osha_rag_metadata.json", "w", encoding="utf-8") as f:
    json.dump(chunk_id_to_metadata, f, ensure_ascii=False, indent=2)

print("✅ OSHA RAG index and metadata saved!")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/109 [00:00<?, ?it/s]

✅ OSHA RAG index and metadata saved!


In [ ]:
# 📂 Load index and metadata
index = faiss.read_index("Osha_GuidelineLLM/osha_rag_index.faiss") #Can change to osha_rag_index.faiss if retrain locally
with open("Osha_GuidelineLLM/osha_rag_metadata.json", "r", encoding="utf-8") as f: ##Can change to osha_rag_metadata.json if retrain locally
    chunk_id_to_metadata = json.load(f)

# 🔁 Load same model used for embedding
model = SentenceTransformer("all-MiniLM-L6-v2")

# 🔍 Retrieval function
def search_osha_guidelines(query, top_k=5):
    query_embedding = model.encode([query])
    D, I = index.search(np.array(query_embedding), top_k)

    results = []
    for idx in I[0]:
        meta = chunk_id_to_metadata[str(idx)]
        results.append({
            "source_file": meta["source_file"],
            "chunk_text": meta["chunk_text"],
            "chunk_id": meta["chunk_id"]
        })

    return results

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def generate_osha_explanation_with_gemini(incident_description, top_chunks):
    context = "\n\n".join(
        [f"From {c['source_file']} (Chunk {c['chunk_id']}):\n{c['chunk_text']}" for c in top_chunks]
    )

    prompt = f"""
A construction incident occurred with the following description:

"{incident_description}"

Below are relevant OSHA guideline excerpts:

{context}

Please explain:
1. What OSHA rules may have been violated?
2. Why the situation was unsafe?
3. What should have been done to prevent it?

Provide your explanation in clear, professional language.
"""

    response = chat.send_message(prompt)
    return response.text


In [ ]:
incident = "A maintenance worker was exposed to chemical fumes while cleaning an industrial tank without proper ventilation."
chunks = search_osha_guidelines(incident, top_k=3)

explanation = generate_osha_explanation_with_gemini(incident, chunks)
print(explanation)

Here's a breakdown of the OSHA violations, unsafe conditions, and preventative measures related to the described incident:

**1. OSHA Rules Potentially Violated:**

Based on the information provided, the following OSHA standards may have been violated:

*   **1910.252(c)(11)(i) Cleaning Compounds - Manufacturer's Instructions:** This standard requires employers to follow the manufacturer's instructions when using cleaning materials. If the manufacturer of the cleaning compound specified the need for ventilation or respiratory protection, failure to follow those instructions is a violation.

*   **1910.146 Permit-Required Confined Spaces (Likely):** While not explicitly stated, cleaning an industrial tank strongly suggests this constitutes a permit-required confined space.  Key elements of this standard that may have been violated include:

    *   **1910.146(c)(1) Permit Program:** The employer failed to implement a written permit space program where a permit-required confined space ex